# Grid-distance comparison

Two open proxies stand in for "is there a network nearby": the LINZ powerline
layer and road centrelines. This notebook asks how much the screening answer
depends on which one you believe, and shows why S-06 is published as two
distances plus a verify flag rather than folded into a score.

**Scope warning.** The committed spatial outputs are deterministic DEMO
geometries with eight surviving fixtures, not Canterbury parcels. Every overlap
number below illustrates the method on n = 8. It is not an empirical estimate
of proxy disagreement, and the notebook prints the sample size beside the
statistic so that it cannot be quoted as one.

In [ ]:
import sys
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))

from nz_solar_siting.grid_distance import compare_top_n
from nz_solar_siting.siting import SitingConfig

candidates = gpd.read_file(ROOT / "outputs" / "demo" / "candidates.gpkg")
print(f"candidates: {len(candidates)} DEMO features (not parcels)")
candidates[["site_id", "grid_line_m", "road_proxy_m", "grid_rank", "road_rank", "rank_shift"]]

## How fast does the shortlist change with N?

A single top-N Jaccard number hides how unstable the comparison is. Sweeping N
shows that the two proxies agree least at the sharp end and converge only as
the shortlist approaches the whole candidate set - which is another way of
saying the ranking carries little information here.

In [ ]:
sweep = pd.DataFrame([compare_top_n(candidates, n) for n in range(2, len(candidates) + 1)])
sweep["sample_note"] = f"demo n={len(candidates)}"
sweep[["n", "overlap_count", "non_overlap_count", "jaccard", "grid_only", "road_only"]]

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.8))
ax.plot(sweep["n"], sweep["jaccard"], marker="o", color="#2d6a9f")
ax.set(
    xlabel="Shortlist size N",
    ylabel="Jaccard overlap",
    ylim=(0, 1.05),
    title=f"Top-N agreement between the two proxies (DEMO, n={len(candidates)})",
)
ax.grid(alpha=0.2)
fig.tight_layout()

## Which sites would the verify flag catch?

`S06_verify_grid` fires when either distance exceeds `grid_distance_review_m`
or the two ranks disagree by at least `rank_shift_review`. Both thresholds live
in `config/assumptions.yml`; the cell below re-derives the flag from the
published distances so the rule can be checked without re-running the pipeline.

In [ ]:
config = SitingConfig()
recomputed = (
    candidates[["grid_line_m", "road_proxy_m"]].max(axis=1) > config.grid_distance_review_m
) | (candidates["rank_shift"] >= config.rank_shift_review)
check = candidates[["site_id", "grid_line_m", "road_proxy_m", "rank_shift", "S06_verify_grid"]].copy()
check["recomputed_flag"] = recomputed
assert check["recomputed_flag"].equals(check["S06_verify_grid"].astype(bool))
print(f"verify-grid flags: {int(recomputed.sum())} of {len(check)}")
check

## Why grid distance is not in `screen_score`

If the two proxies disagree this much, a weighted score built on them would
hand a reviewer a confident-looking ordering derived from the least reliable
input. `screen_score` therefore uses only solar resource and usable area, and
grid proximity is carried as two published distances plus a flag. Moving the
powerline layer changes the distances and leaves the ordering untouched.

In [ ]:
from nz_solar_siting.demo_data import build_demo_layers
from nz_solar_siting.siting import evaluate_sites

sites, conservation, powerlines, roads = build_demo_layers()
baseline, _ = evaluate_sites(sites, conservation, powerlines, roads)
moved = powerlines.copy()
moved["geometry"] = moved.geometry.translate(xoff=50_000.0)
shifted, _ = evaluate_sites(sites, conservation, moved, roads)

comparison = pd.DataFrame({
    "site_id": baseline["site_id"],
    "grid_line_m": baseline["grid_line_m"],
    "grid_line_m_after_move": shifted["grid_line_m"],
    "screen_score": baseline["screen_score"],
    "screen_score_after_move": shifted["screen_score"],
})
print("score unchanged:", comparison["screen_score"].equals(comparison["screen_score_after_move"]))
comparison

## The same comparison on real geometry

Everything above is a method demonstration on twelve rectangles. LINZ Topo50 and
LCDB need portal accounts, so the repository also ships an OpenStreetMap
substitute for the Canterbury plains: mapped farmland polygons, mapped
transmission and distribution lines, and road centrelines. The rules that OSM can
support - S-01 area and S-02 width - leave a sample in the thousands, which is
enough for the comparison to mean something.

OSM is not LINZ: completeness varies by area and contributor, and a mapped
land-use polygon is a land-use observation, not a parcel title. Read what follows
as an OSM measurement.

In [ ]:
import json

from nz_solar_siting.osm_layers import read_osm_layers

farmland, powerlines, roads = read_osm_layers(ROOT / "data" / "derived" / "osm")
study = json.loads((ROOT / "outputs" / "osm" / "osm_grid_study.json").read_text(encoding="utf-8"))
print(f"farmland polygons downloaded : {study['farmland_polygons_downloaded']:,}")
print(f"passing area and width       : {study['sites_passing_area_and_width']:,}")
study["network_features"], study["median_distance_m"]

## Three proxies, three different answers

"Distance to the network" has no single meaning. Transmission is the population a
utility-scale connection actually cares about; distribution is everywhere;
road centrelines are an access proxy that people reach for because the layer is
easy to get. The rank correlations say how interchangeable they are.

In [ ]:
osm = pd.read_csv(ROOT / "outputs" / "osm" / "osm_grid_distance.csv")
pairs = pd.DataFrame({
    "rank_correlation": study["rank_correlation"],
    "median_rank_shift": study["median_rank_shift"],
    "rank_shift_p90": study["rank_shift_p90"],
})
pairs["top50_jaccard"] = {
    pair: next(row["jaccard"] for row in sweep if row["n"] == 50)
    for pair, sweep in study["top_n_sweep"].items()
}
pairs

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
for pair, sweep in study["top_n_sweep"].items():
    frame = pd.DataFrame(sweep)
    ax.plot(frame["n"], frame["jaccard"], marker="o", label=pair.replace("_m", "").replace("|", " vs "))
ax.set(
    xlabel="Shortlist size N",
    ylabel="Jaccard overlap",
    ylim=(0, 1.02),
    title=f"Real OSM geometry, n = {len(osm):,}",
)
ax.legend(frameon=False, fontsize=9)
ax.grid(alpha=0.2)
fig.tight_layout()

## The aerial step is not optional

The twenty largest transmission-versus-road disagreements are queued with
coordinates and a LINZ Basemaps deep link. Three have been reviewed and logged.
The most instructive one is the polygon the road proxy ranks first: it is a
coastal sand spit tagged `landuse=meadow`, bare gravel and dune, not developable
land at all. No distance calculation would have caught that.

In [ ]:
queue = pd.read_csv(ROOT / "outputs" / "osm" / "aerial_review_queue.csv")
reviewed = queue[queue["finding"].notna()]
print(f"queued {len(queue)}, reviewed {len(reviewed)}")
for _, row in reviewed.iterrows():
    print(f"\n{row.site_id}  transmission {row.transmission_m:,.0f} m / road {row.road_m:,.0f} m")
    print(f"  {row.finding}")

## What is still missing

The LINZ Topo50 and LCDB versions of this same run, and the remaining seventeen
aerial checks. Both are blocked on portal credentials rather than on code.
Nothing here should be quoted as a LINZ result.

Attribution: (c) OpenStreetMap contributors, ODbL 1.0. Aerial imagery (c) LINZ
and Environment Canterbury, CC BY 4.0.